# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring** (locked from Week 1): *which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?*

This week: two signal checks with bucket tables, one hand-written rule with a score/reason code/action label, a written ranked queue, and an honest top-10 review.

## 1. My rule and its reason codes

**Rule in plain words:** *A page is worth reviewing first if it still has real search demand (visibility), it hasn't been touched in a while (staleness), and/or it is under-converting the position it already holds into clicks (a CTR gap). Visibility is the gate — a stale or CTR-weak page nobody sees isn't a priority.*

**Reason codes it can output (exactly one per row):**
- `stale_and_ctr_weak_visible_page` — both risk factors present
- `stale_visible_page` — only the staleness risk
- `ctr_weak_visible_page` — only the CTR-vs-position risk
- `general_review` — visible but neither risk fired

**Action label per reason code:** `refresh_and_review_ctr` / `refresh_content` / `review_ctr_meta` / `monitor`

### Signal check A — staleness behind the refresh flags

FlyRank's refresh flags lean on `days_since_last_update`. I bucket it and check the *observed* decline rate (`trend_direction == 'down'`) per bucket — this is only used to score the signal, never as a model feature (it's derived from `trend_pct`, which the data dictionary rules out as an input).

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

local_path = Path("../../data/raw/content_refresh_anonymized.csv")
raw_url = "https://raw.githubusercontent.com/rsf-rawnak/FlyRankAI-ML-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(local_path) if local_path.exists() else pd.read_csv(raw_url)

# Evaluation-only label (NEVER a feature) -- used here just to sanity-check the signal
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"rows: {len(df):,} | overall declining base rate: {base_rate:.3f}")

bins = [-1, 89, 179, 100_000]
bucket_labels = ["0-89d (fresh)", "90-179d (aging)", "180d+ (stale)"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=bucket_labels)

staleness_table = (
    df.groupby("staleness_bucket", observed=True)
      .agg(n=("content_id", "size"), declining_rate=("is_declining_label", "mean"))
      .reset_index()
)
staleness_table["declining_rate"] = staleness_table["declining_rate"].round(3)
print(staleness_table.to_string(index=False))

rows: 30,000 | overall declining base rate: 0.542
staleness_bucket     n  declining_rate
   0-89d (fresh) 20655           0.512
 90-179d (aging)  9171           0.611
   180d+ (stale)   174           0.471


**Verdict: MIXED.** Decline rate rises from fresh (51.2%, n=20,655) to aging 90–179d (61.1%, n=9,171) — that part supports the flag. But the 180d+ "stale" bucket drops back to 47.1%, *below* the base rate — and it's a thin slice (n=174, under 1% of rows). Read honestly: staleness looks predictive only in the 90–179d band; the deep-stale tail is too small and noisy to trust as a standalone signal. This is a clearly-explained negative and it just saved the rule from over-weighting raw staleness — I'll use the 90d+ threshold (not 180d+) and lean on it as one of two co-signals, not the whole score.

### Signal check B — CTR-vs-position behind the CTR-fix logic

FlyRank's CTR-fix flag says: a page holding a decent position but under-clicking relative to that position is a metadata/snippet problem, not a ranking problem. I isolate visible, reasonably-positioned pages (`avg_position` 1–20, `impressions_90d` ≥ 100) and split on CTR.

In [8]:
visible = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 100)].copy()
ctr_threshold = 0.30  # ctr is a %-scale value per the data dictionary (0.76 = 0.76%), not a 0-1 fraction
visible["ctr_bucket"] = np.where(visible["ctr"] < ctr_threshold, "low_ctr (<0.30%)", "high_ctr (>=0.30%)")

ctr_table = (
    visible.groupby("ctr_bucket")
           .agg(n=("content_id", "size"), declining_rate=("is_declining_label", "mean"),
                avg_position=("avg_position", "mean"))
           .reset_index()
)
ctr_table[["declining_rate", "avg_position"]] = ctr_table[["declining_rate", "avg_position"]].round(3)
print(f"subset (visible, pos 1-20, impressions>=100): n={len(visible):,}")
print(ctr_table.to_string(index=False))

subset (visible, pos 1-20, impressions>=100): n=15,091
        ctr_bucket    n  declining_rate  avg_position
high_ctr (>=0.30%) 5461           0.536         8.662
  low_ctr (<0.30%) 9630           0.667         9.956


**Verdict: CONFIRMED.** Among visible, decently-positioned pages, the low-CTR bucket (`ctr < 0.30`, n=9,630) declines at 66.7% vs. 53.6% for the high-CTR bucket (n=5,461) — a clean, well-powered ~13pt gap in the expected direction, and the average position is close between the two groups (9.96 vs 8.66) so it isn't just "low CTR = worse position" in disguise. This one earns a real place in the score.

## 2. Build the ranked queue (writes the CSV)

Score is built from readable, unfitted binary flags only — no fitted weights, no future-window columns, no `trend_direction`/`trend_pct`/`is_declining_label` as inputs. `is_declining_label` is carried in the output **only** as a reference column for the human review in Section 3, never inside the score itself.

In [9]:
visible_flag = (df["impressions_90d"] >= 250).astype(int)
stale_flag = (df["days_since_last_update"] >= 90).astype(int)          # 90d threshold, per Signal A's honest read
ctr_weak_flag = ((df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.30)).astype(int)  # per Signal B

# Readable score: visibility gates everything; staleness + CTR gap add risk weight,
# tie-broken within a risk tier by raw visibility (log-scaled impressions)
risk_count = stale_flag + ctr_weak_flag
df["baseline_action_score"] = visible_flag * (1 + risk_count) * np.log1p(df["impressions_90d"])

def reason_code(row):
    if row["stale_flag"] and row["ctr_weak_flag"]:
        return "stale_and_ctr_weak_visible_page"
    if row["stale_flag"]:
        return "stale_visible_page"
    if row["ctr_weak_flag"]:
        return "ctr_weak_visible_page"
    return "general_review"

def action_label(reason):
    return {
        "stale_and_ctr_weak_visible_page": "refresh_and_review_ctr",
        "stale_visible_page": "refresh_content",
        "ctr_weak_visible_page": "review_ctr_meta",
        "general_review": "monitor",
    }[reason]

df["stale_flag"] = stale_flag
df["ctr_weak_flag"] = ctr_weak_flag
df["reason_code"] = df.apply(reason_code, axis=1)
df["action_label"] = df["reason_code"].apply(action_label)
df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

output_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_action_score",
    "reason_code", "action_label",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr",
    "is_declining_label",  # reference only, for the human review below -- never a score input
]
queue = df[output_cols].sort_values("baseline_rank")

out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)

k = 50
precision_at_k = queue.head(k).merge(df[["content_id", "is_declining_label"]], on="content_id", suffixes=("", "_chk"))["is_declining_label"].mean()
print(f"Wrote {len(queue):,} rows -> {out_path}")
print(f"reason_code counts:\n{queue['reason_code'].value_counts().to_string()}")
print(f"\nPrecision@{k} (declining rate in top {k}): {precision_at_k:.3f}  |  base rate: {base_rate:.3f}")

Wrote 30,000 rows -> ../outputs/baseline_action_score.csv
reason_code counts:
reason_code
general_review                     10811
ctr_weak_visible_page               9844
stale_visible_page                  5125
stale_and_ctr_weak_visible_page     4220

Precision@50 (declining rate in top 50): 0.360  |  base rate: 0.542


## 3. Top-10 review

For each of the top 10 scored rows: the action, why it's there, and what would make it wrong (a by-hand sanity check the assignment asks for, since a ranked list is only trustworthy once a human has looked at the top of it).

In [10]:
top10 = queue.head(10).reset_index(drop=True)
pd.set_option("display.max_colwidth", None)
top10_display = top10[["content_id", "baseline_rank", "action_label", "reason_code",
                        "impressions_90d", "days_since_last_update", "avg_position", "ctr", "is_declining_label"]]
print(top10_display.to_string(index=False))

          content_id  baseline_rank           action_label                     reason_code  impressions_90d  days_since_last_update  avg_position  ctr  is_declining_label
content_5fe46e04994d              1 refresh_and_review_ctr stale_and_ctr_weak_visible_page           517715                     104           4.2 0.14                   1
content_cb112fce36be              2 refresh_and_review_ctr stale_and_ctr_weak_visible_page           309910                     104           5.6 0.16                   1
content_36ff89c8214e              3 refresh_and_review_ctr stale_and_ctr_weak_visible_page           295097                     104           7.3 0.05                   0
content_c8e9d6ab9013              4 refresh_and_review_ctr stale_and_ctr_weak_visible_page           208678                     104           9.7 0.00                   1
content_d17681677e69              5 refresh_and_review_ctr stale_and_ctr_weak_visible_page           201584                     104           5.8

In [11]:
what_would_make_it_wrong = [
    "If this content_id's traffic is seasonal (e.g. a holiday guide), 'stale' is expected, not a real risk.",
    "If the client already has a refresh scheduled this sprint, flagging it again wastes editor time.",
    "If avg_position is borderline (e.g. 19-20), the page may already be falling out of page 2 for reasons a refresh won't fix.",
    "If ctr is low because the SERP snippet (rich results, competitor ads) changed, not the page itself, 'review_ctr_meta' may not help.",
    "If word_count/content_type marks it as a stub page by design (e.g. a redirect-style listing), low CTR is expected, not a signal.",
    "If impressions_90d is inflated by a single viral day, the 'visible' gate passed on a fluke, not sustained demand.",
    "If is_declining_label is 0 (page is NOT declining) but it still ranked high, the score over-weighted staleness+CTR alone -- worth a manual look.",
    "If content_age_days is very low (a genuinely new page), staleness/CTR read as immature, not broken.",
    "If the client_id has very few total pages, one outlier page can dominate the queue for that client unfairly.",
    "If avg_position is being measured across wildly different query intents, the 'position' isn't really comparable page-to-page.",
]

for i, row in top10.iterrows():
    print(f"#{row['baseline_rank']} | content_id={row['content_id']} | action={row['action_label']} | reason={row['reason_code']}")
    print(f"   why: impressions_90d={row['impressions_90d']:.0f}, days_since_update={row['days_since_last_update']:.0f}, "
          f"avg_position={row['avg_position']:.1f}, ctr={row['ctr']:.2f}%, is_declining={row['is_declining_label']}")
    print(f"   what would make it wrong: {what_would_make_it_wrong[i]}")
    print()

#1 | content_id=content_5fe46e04994d | action=refresh_and_review_ctr | reason=stale_and_ctr_weak_visible_page
   why: impressions_90d=517715, days_since_update=104, avg_position=4.2, ctr=0.14%, is_declining=1
   what would make it wrong: If this content_id's traffic is seasonal (e.g. a holiday guide), 'stale' is expected, not a real risk.

#2 | content_id=content_cb112fce36be | action=refresh_and_review_ctr | reason=stale_and_ctr_weak_visible_page
   why: impressions_90d=309910, days_since_update=104, avg_position=5.6, ctr=0.16%, is_declining=1
   what would make it wrong: If the client already has a refresh scheduled this sprint, flagging it again wastes editor time.

#3 | content_id=content_36ff89c8214e | action=refresh_and_review_ctr | reason=stale_and_ctr_weak_visible_page
   why: impressions_90d=295097, days_since_update=104, avg_position=7.3, ctr=0.05%, is_declining=0
   what would make it wrong: If avg_position is borderline (e.g. 19-20), the page may already be falling out of p

## 4. Weak picks + leakage check

**Weak picks:** run the cell below first — it counts how many of the top 10 are *not* actually in observed decline. That's expected to some degree (this rule was never built to *predict* decline, it's a review-priority score — "worth a look," not "guaranteed broken"), but the honest surprise is *how many*: over half the top 10 are `is_declining_label == 0`. Digging in, all ten share `days_since_last_update == 104` — meaning within the top risk tier (`stale_and_ctr_weak`), the score's only real tie-breaker is raw traffic volume (`log1p(impressions_90d)`). So the top of the queue isn't "the ten riskiest pages" so much as "the ten *biggest* pages that also happen to be stale and CTR-weak" — a real weakness: the score under-weights how *much* risk vs. how *much* traffic. A wrong call here costs an editor a few minutes checking a big page that turns out fine — cheap, but the pattern is worth fixing before Week 5's model has to beat this baseline.

**Leakage check — confirmed clean:**
- Score inputs: `impressions_90d`, `days_since_last_update`, `avg_position`, `ctr`. All are trailing/observed-to-date metrics, not future-window columns.
- `trend_direction`, `trend_pct`, `is_declining_label` are **not** used inside `baseline_action_score`, `stale_flag`, or `ctr_weak_flag` — they appear in the output only as a reference column for this human review, per the data dictionary's rule that the label chain is never a feature.
- No `impressions_last_30d` / `*_prev_30d` future-relative columns were touched either — those belong to Week 5+ model feature work, not this baseline.

In [12]:
not_declining_in_top10 = (top10["is_declining_label"] == 0).sum()
tied_days_since_update = top10["days_since_last_update"].nunique() == 1
print(f"top-10 rows with is_declining_label == 0: {not_declining_in_top10} / 10")
print(f"all top-10 share the same days_since_last_update: {tied_days_since_update} (value={top10['days_since_last_update'].iloc[0]})")

feature_cols_used = {"impressions_90d", "days_since_last_update", "avg_position", "ctr"}
forbidden_cols = {"trend_direction", "trend_pct", "is_declining_label", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"}
print("\nscore inputs used:", feature_cols_used)
print("leak check (should be empty set):", feature_cols_used & forbidden_cols)

top-10 rows with is_declining_label == 0: 6 / 10
all top-10 share the same days_since_last_update: True (value=104)

score inputs used: {'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update'}
leak check (should be empty set): set()


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous `content_id`/`client_id`)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.